> **Companion notebook** — generated from `modules/08-frequentist-bridge.md` (the canonical, harness-verified text; regenerate with `python tools/make_notebooks.py`). Run cells top-to-bottom from the `curriculum/` directory so `figures/...` paths resolve. Cells marked *illustration only* are intentionally not executable.

# 08. The Frequentist Bridge: Stein as Hero  [SIGNATURE S4]

> **Spine.** The two theories fuse asymptotically (Bernstein–von Mises); where they don't, shrinkage wins — the frequentist crown jewel, the estimator that dominates the MLE, is a Bayesian posterior mean in disguise.
> **Which line?** All four, viewed from outside. This module audits the machine of modules 00–07 against the *sampling distribution* — the frequentist's ledger — and finds the Bayesian procedures pass, and sometimes win.
> **Promise.** After this module you can state exactly when a credible interval is also a confidence interval, explain why the "obviously optimal" MLE is inadmissible in $d\ge 3$, derive James–Stein as empirical Bayes, and read frequentist evaluation as an audit your models should welcome rather than a rival creed.
> **Prereqs.** Modules 04 (likelihood, Fisher information, MLE), 05 (conjugate updating, shrinkage formula), 06 (estimates are decisions, coverage, ancillarity).
> **Runtime.** ~5 s.
> **Sources.** C-B §7.3 (Cramér–Rao, loss/risk, Bayes rules), §10.1 (MLE asymptotics); Efron–Morris on James–Stein by concept; booklet ch. 9 (shrinkage) preview.

Here is a belief almost every trained reader holds, and it is *almost* right. You want to estimate the mean $\theta$ of a Gaussian. You take the sample mean $\bar x$ — the MLE. It is unbiased. Its variance attains the Cramér–Rao bound, so no unbiased estimator beats it. Asymptotically it is efficient, and by Bernstein–von Mises the flat-prior posterior agrees with it to leading order. *The MLE for a Gaussian mean is obviously optimal.* Every clause of that sentence is a theorem. And the conclusion is false the moment you estimate **three means at once**.

First, watch the MLE earn every word of that reputation. The mood throughout is not "gotcha." It is: *their best estimator is our shrinkage.*

In [ ]:
# --- setup ---
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

SLUG = "08-frequentist-bridge"           # this module's figure dir
FIG = Path("figures") / SLUG
FIG.mkdir(parents=True, exist_ok=True)
SEED = 0
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (7, 4), "figure.dpi": 110, "savefig.dpi": 150,
    "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11,
})

def save(fig, name):
    out = FIG / f"{name}.png"
    fig.savefig(out)
    plt.close(fig)
    print(f"[fig] {out}")

## 08.1 The sampling distribution, and everything the MLE gets right

Frequentist evaluation asks a single question about a procedure $\hat\theta(\cdot)$: **if the true $\theta$ held and I reran the whole experiment forever, how would $\hat\theta$ behave?** The answer is the *sampling distribution* of $\hat\theta$ — a distribution over datasets, holding $\theta$ fixed. This is the mirror image of the Bayesian question (fix the data, vary $\theta$), and keeping the two straight is the whole module.

The marquee frequentist result about MLEs (module 04's forward promise, C-B Theorem 10.1.12): under regularity, the MLE is consistent and **asymptotically efficient**,
$$\sqrt{n}\,(\hat\theta_n - \theta_0)\ \xrightarrow{d}\ N\!\big(0,\ \mathcal I(\theta_0)^{-1}\big),$$
so for large $n$ the sampling distribution is $\hat\theta_n \approx N(\theta_0,\ 1/(n\mathcal I(\theta_0)))$ — centered at the truth, with variance the inverse Fisher information. Watch it hold for the Poisson mean, where $\mathcal I(\lambda)=1/\lambda$ so the asymptotic variance is $\lambda/n$.

In [ ]:
# Sampling distribution of the MLE (many datasets) vs its Gaussian limit.
lam0, n_pois, R = 4.0, 30, 200_000
# xbar over R datasets; sum of n Poisson(lam) is Poisson(n*lam), so draw directly.
xbar_pois = rng.poisson(lam0 * n_pois, size=R) / n_pois
sd_sim = xbar_pois.std()
sd_asym = np.sqrt(lam0 / n_pois)                     # sqrt(I^-1 / n), I(lam)=1/lam
ks = stats.kstest(xbar_pois, stats.norm(lam0, sd_asym).cdf).statistic
print(f"Poisson MLE xbar:  sim sd = {sd_sim:.4f}   asymptotic sqrt(lam/n) = {sd_asym:.4f}")
print(f"KS(sampling dist, N(lam, lam/n)) = {ks:.4f}")

fig, ax = plt.subplots()
grid = np.linspace(lam0 - 4*sd_asym, lam0 + 4*sd_asym, 300)
ax.hist(xbar_pois, bins=60, density=True, color="C7", alpha=0.6, label="sampling dist of MLE (sim)")
ax.plot(grid, stats.norm(lam0, sd_asym).pdf(grid), "k--", lw=2, label="N(λ, λ/n) asymptotic")
ax.axvline(lam0, color="C3", lw=1, label="truth λ=4")
ax.set_xlabel("MLE  $\\hat\\lambda=\\bar x$"); ax.set_ylabel("density")
ax.set_title("MLE asymptotics: sampling distribution ≈ N(θ, I⁻¹/n)")
ax.legend(fontsize=9)
save(fig, "mle-asymptotics")

![Histogram of the Poisson sample mean over 200,000 datasets, overlaid with the N(4, 4/30) asymptotic Gaussian; the two coincide.](figures/08-frequentist-bridge/mle-asymptotics.png)

The simulated sampling-distribution standard deviation is `0.3653`, matching the asymptotic $\sqrt{\lambda/n}=$ `0.3651` to three figures; the KS distance to the Gaussian is `0.0256`. The MLE is doing exactly what the theorem says.

**Cramér–Rao, and its escape hatch.** For *unbiased* estimators there is a hard floor (C-B Theorem 7.3.9): any unbiased $W$ with $E_\theta W=\theta$ satisfies $\mathrm{Var}_\theta[W]\ge 1/(n\mathcal I(\theta))$, the **Cramér–Rao lower bound (CRLB)**. The proof is one Cauchy–Schwarz step on $\mathrm{Cov}[W,\text{score}]$. For the Gaussian mean the sample mean attains it exactly.

In [ ]:
# Cramér-Rao: xbar attains the bound 1/(nI) for the Normal mean (I=1/sigma^2).
mu, sig2, n_cr, R = 2.0, 1.0, 20, 300_000
xbar_norm = rng.normal(mu, np.sqrt(sig2), size=(R, n_cr)).mean(1)
crlb = sig2 / n_cr                                   # 1/(nI) with I = 1/sigma^2
print(f"CRLB 1/(nI) = {crlb:.5f}     Var(xbar) sim = {xbar_norm.var():.5f}")

The bound is `0.05000` and the sample mean's simulated variance is `0.04997`: the Gaussian mean is a *best unbiased* estimator, no unbiased competitor can do better at any $\theta$. But read the proviso twice. The bound governs **unbiased** estimators whose support does not depend on $\theta$. The uniform $U(0,\theta)$ violates the support clause (C-B Example 7.3.13) — its MLE $\max_i x_i$ has variance *below* the naive CRLB because the theorem simply does not apply. And the entire escape we are about to exploit is the first clause: **a biased estimator is not bound by the CRLB at all.** Stein walked straight through that door.

## 08.2 Stein's paradox: three unrelated quantities  [SIGNATURE S4]

**Setup.** You must estimate three completely unrelated numbers: a baseball player's true batting average, the mean wheat yield of a farm district, and the fuel economy of a car model. Standardize each so its measurement is $X_i \sim N(\theta_i, 1)$ — you observe each quantity once, with unit noise. The obvious estimator is $\hat\theta_i = X_i$: use the batting measurement for batting, the yield measurement for yield. This is the MLE, coordinate by coordinate, and by §08.1 each coordinate is individually optimal.

**Predict.** Before any code, commit to an answer. *Could you ever lower your total squared error by letting the baseball number influence your wheat estimate?* The overwhelming intuition — say it out loud — is **obviously not; the quantities are unrelated, so borrowing strength across them can only inject noise.** The naive reasoning: each $X_i$ is the best estimate of its own $\theta_i$, and independent problems should be solved independently. Hold that "no."

**Run.** Total loss is $\sum_i(\hat\theta_i-\theta_i)^2$. The **James–Stein** estimator shrinks the whole vector toward zero by a data-chosen factor,
$$\delta^{\mathrm{JS}}(X) = \Big(1 - \frac{d-2}{\lVert X\rVert^2}\Big)X,\qquad
\delta^{\mathrm{JS+}}(X) = \Big(1 - \frac{d-2}{\lVert X\rVert^2}\Big)_{\!+}X,$$
where $(\cdot)_+$ is the positive part (never flip past zero). Take $d=10$ unrelated quantities with every $\theta_i=0$ and compare total MSE over many replications.

In [ ]:
# Stein's paradox: total MSE of MLE vs James-Stein, d=10 unrelated means at theta=0.
def js_mse(theta, R=40_000, gen=None):
    gen = gen or rng
    d = len(theta)
    X = gen.normal(theta, 1.0, size=(R, d))          # X_i ~ N(theta_i, 1)
    sq = (X**2).sum(1, keepdims=True)
    shrink = 1 - (d - 2) / sq                         # James-Stein factor
    js  = shrink * X
    jsp = np.maximum(shrink, 0.0) * X                 # positive-part JS
    err = lambda est: ((est - theta)**2).sum(1).mean()
    return err(X), err(js), err(jsp)

d = 10
m_mle, m_js, m_jsp = js_mse(np.zeros(d))
print(f"d={d}, theta=0:  MSE(MLE) = {m_mle:.2f}   MSE(JS) = {m_js:.2f}   MSE(JS+) = {m_jsp:.2f}")

The MLE's total MSE is `9.98` — essentially $d=10$, one unit of error per coordinate, as it must be. The James–Stein estimator posts `1.99`, and its positive-part cousin `1.25`. **A fivefold reduction in total error, achieved by letting each unrelated measurement pull the others toward a common center.** The naive "no" is wrong.

**Reconcile — is this a fluke of $\theta=0$?** No. Stein's theorem says $\delta^{\mathrm{JS}}$ has *strictly smaller* risk than the MLE at **every** $\theta$ when $d\ge 3$ — it **dominates**. The gain is largest when the $\theta_i$ cluster near the shrinkage target and shrinks toward zero as they spread out, but it never goes negative. Sweep the signal size $\lVert\theta\rVert$ and watch.

In [ ]:
# Dominance: JS+ risk stays below the MLE's flat risk of d=10 across ||theta||.
d = 10
norm_grid = np.array([0., 1., 2., 3., 4., 6., 8., 12.])
risk_mle, risk_jsp = [], []
for nm in norm_grid:
    th = np.zeros(d); th[0] = nm                      # put all signal on one axis
    a, _, c = js_mse(th, R=20_000)
    risk_mle.append(a); risk_jsp.append(c)
risk_mle, risk_jsp = np.array(risk_mle), np.array(risk_jsp)
print("||theta||:", norm_grid.astype(int))
print("MLE risk :", np.round(risk_mle, 2))
print("JS+ risk :", np.round(risk_jsp, 2))

fig, ax = plt.subplots()
ax.plot(norm_grid, risk_mle, "o-", color="C0", label="MLE (X)  — flat at d=10")
ax.plot(norm_grid, risk_jsp, "s-", color="C1", label="James–Stein⁺")
ax.axhline(d, color="C0", ls=":", lw=1)
ax.set_xlabel("‖θ‖  (distance of the truth from the shrinkage target)")
ax.set_ylabel("total risk  E‖δ − θ‖²")
ax.set_title("Stein dominance (d=10): JS⁺ risk is below the MLE's everywhere")
ax.legend()
save(fig, "stein-dominance")

![Risk versus signal size for d=10: the MLE risk is flat at 10 while the James–Stein-plus risk rises from about 1.3 toward 10 but stays strictly below it.](figures/08-frequentist-bridge/stein-dominance.png)

The MLE risk is flat at $d=10$ regardless of $\lVert\theta\rVert$ (the sampling distribution of $X$ does not care where $\theta$ sits). James–Stein$^+$ climbs from `1.28` at the origin (the same quantity as the `1.25` above — two independent Monte-Carlo runs) toward 10 as the signal grows, but never touches it — biased everywhere, better everywhere. That is what *dominance* means, and it is why the MLE is **inadmissible** in $d\ge3$: a rule you should never use, because another rule beats it at every single $\theta$.

**The collapse at low dimension.** The magic needs $d\ge 3$. The factor $d-2$ is the tell.

In [ ]:
# Dimension collapse: JS gains nothing until d>=3.
for dd in (2, 3):
    a, _, c = js_mse(np.zeros(dd), R=40_000)
    print(f"d={dd}, theta=0:  MSE(MLE) = {a:.2f}   MSE(JS+) = {c:.2f}")

At $d=2$ the factor is $d-2=0$: James–Stein *is* the MLE, both scoring `1.99`, no improvement possible — the MLE is admissible. At $d=3$ the shrinkage switches on and JS$^+$ drops to `1.62` against the MLE's `3.01`. (At $d=1$ the formula degenerates entirely, inflating rather than shrinking; the one-parameter MLE of §08.1 is safe.) The crossover into paradox is exactly at $d=3$. Three is the smallest number of unrelated problems you can couple to your advantage.

## 08.3 James–Stein is an empirical-Bayes posterior mean

Where did that estimator come from, and why $d-2$? Put a prior on the unrelated means — the very thing the naive view forbade — and the formula assembles itself. Suppose $\theta_i \sim N(0,\tau^2)$ and $X_i\mid\theta_i \sim N(\theta_i,1)$. This is module 05's Normal–Normal conjugate pair, one per coordinate. The posterior mean is the master shrinkage formula from module 05: a precision-weighted blend of the data $X_i$ and the prior mean $0$,
$$E[\theta_i\mid X_i] = \Big(\underbrace{\tfrac{\tau^2}{1+\tau^2}}_{\text{data weight}}\Big)X_i = \big(1 - B\big)X_i,\qquad B = \frac{1}{1+\tau^2}.$$
The shrinkage fraction $B$ depends on $\tau^2$, which we pretend not to know — so **estimate it from the data**. Marginally, integrating out $\theta_i$, each $X_i \sim N(0,\,1+\tau^2)$, so $\lVert X\rVert^2 \sim (1+\tau^2)\,\chi^2_d$. Because $E[1/\chi^2_d]=1/(d-2)$,
$$E\!\left[\frac{d-2}{\lVert X\rVert^2}\right] = \frac{d-2}{(1+\tau^2)(d-2)} = \frac{1}{1+\tau^2} = B.$$
So $\widehat B = (d-2)/\lVert X\rVert^2$ is an **unbiased estimate of the shrinkage fraction**, and plugging it into the posterior mean $(1-B)X$ gives exactly $\delta^{\mathrm{JS}} = (1 - (d-2)/\lVert X\rVert^2)X$. Verify the identity numerically:

In [ ]:
# James-Stein = empirical Bayes: (d-2)/||X||^2 estimates the shrinkage fraction B=1/(1+tau^2).
d = 10
print("tau^2 | true B=1/(1+tau^2) | E[(d-2)/||X||^2]")
for tau2 in (0.5, 1.0, 3.0):
    X = rng.normal(0.0, np.sqrt(1 + tau2), size=(200_000, d))   # marginal N(0,1+tau^2)
    Bhat = ((d - 2) / (X**2).sum(1)).mean()
    print(f"  {tau2:>3} |        {1/(1+tau2):.4f}        |      {Bhat:.4f}")

The estimated shrinkage fraction tracks the truth closely: `0.6667`/`0.6654` at $\tau^2=0.5$, `0.5000`/`0.4994`, `0.2500`/`0.2503`. James–Stein is the Normal–Normal posterior mean with its prior variance read off the marginal likelihood of the same data — the first **empirical Bayes** estimator, and the reason the whole idea has a name. Here is the punchline worth memorizing: **admissibility forced frequentists to invent a posterior mean.** The estimator that a purely frequentist criterion (dominate the MLE) demands turns out to *be* the Bayesian answer under a prior nobody had to believe in advance. Module 16 makes this adaptive across a genuine hierarchy; here it is the bridge's keystone.

## 08.4 Bernstein–von Mises: why they agreed, and where they don't

Why did the paradigms agree so perfectly for one Gaussian mean in §08.1, and disagree so violently in §08.2? The one-parameter agreement is a theorem, and it has a name.

**Bernstein–von Mises (BvM), stated.** Fix a well-specified model with a finite-dimensional parameter. As $n\to\infty$, the posterior distribution of $\theta$, *centered at the MLE and scaled by $\sqrt n$*, converges to the same Gaussian as the sampling distribution of the MLE:
$$p(\theta\mid y)\ \approx\ N\!\big(\hat\theta_n,\ \mathcal I_n^{-1}\big)\ \approx\ \text{(sampling law of }\hat\theta_n\text{)}.$$
Consequence: the Bayesian's credible interval and the frequentist's confidence interval **coincide asymptotically**. The prior washes out; the likelihood's curvature (Fisher information, module 04) sets both widths. Watch it converge for a skewed model where the finite-$n$ posterior is *visibly* non-Gaussian — the rate of an exponential, whose Gamma posterior is right-skewed at small $n$. **Predict** first: at $n=5$, with a near-flat prior, are the centered posterior and the centered sampling law of the MLE already the same curve — total-variation distance near zero — or visibly different? The tempting shortcut: "flat prior means posterior $=$ likelihood $=$ what the MLE sees, so they should match at any $n$."

In [ ]:
# BvM: overlay the posterior (ONE dataset) on the sampling dist of the MLE (MANY datasets).
lam0 = 1.0                                            # true rate; Exp(rate), I(lam)=1/lam^2
a0 = b0 = 0.001                                       # near-flat Gamma prior on the rate
M = 200_000

def tv(a, b, bins=60):                                # total-variation distance via histograms
    lo, hi = min(a.min(), b.min()), max(a.max(), b.max())
    e = np.linspace(lo, hi, bins + 1)
    pa, _ = np.histogram(a, e, density=True); pb, _ = np.histogram(b, e, density=True)
    return 0.5 * np.abs(pa - pb).sum() * (e[1] - e[0])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, n in zip(axes, (5, 50, 500)):
    y = rng.exponential(1 / lam0, size=n); S = y.sum(); lhat = n / S     # one dataset, its MLE
    post = rng.gamma(a0 + n, 1 / (b0 + S), size=M)                       # posterior on the rate
    mle_samp = n / rng.gamma(n, 1 / lam0, size=M)                        # MLE sampling dist (Σy~Gamma)
    # BvM is a claim about SHAPE/WIDTH: center each at its OWN mean, then compare.
    d_tv = tv(post - post.mean(), mle_samp - mle_samp.mean())
    skew = stats.skew(post)
    print(f"BvM n={n:3d}:  MLE={lhat:.3f}  TV(centered post, MLE-samp) = {d_tv:.4f}  post skew = {skew:.3f}")
    ax.hist(mle_samp, bins=70, density=True, color="C7", alpha=0.6, label="MLE sampling dist")
    ax.hist(post, bins=70, density=True, histtype="step", color="C1", lw=2, label="posterior")
    ax.axvline(lam0, color="C3", lw=1); ax.set_title(f"n={n}   TV={d_tv:.3f}")
    ax.set_xlabel("rate λ")
axes[0].legend(fontsize=8); axes[0].set_ylabel("density")
fig.suptitle("Bernstein–von Mises: posterior (1 dataset) fuses with MLE sampling law (many)")
fig.tight_layout(rect=(0, 0, 1, 0.93))
save(fig, "bvm-convergence")

![Three panels for n=5, 50, 500: at n=5 the posterior and MLE sampling distributions are skewed; by n=500 both are the same narrow Gaussian, TV shrinking from 0.17 to 0.02.](figures/08-frequentist-bridge/bvm-convergence.png)

The centered total-variation distance between the posterior and the MLE's sampling law falls from `0.1668` at $n=5$ to `0.1446` at $n=50$ to `0.0165` at $n=500$, while the posterior skew decays from `0.886` toward zero. So the flat-prior shortcut fails at small $n$: the two laws answer different questions (vary $\theta$ vs vary data) and merely *share a likelihood*; only the asymptotics fuse them. By $n=500$ they are the same narrow Gaussian, skew gone. (The centering hides the theorem's other half: the posterior mean and the MLE also coincide to leading order — their gap is $O(1/n)$ — so centering at either is asymptotically the same choice; this demo isolates the shape-and-width claim on purpose.) **This is the deep reason module 04's Fisher-normal interval matched the exact Beta interval: BvM.** It is also the license, used everywhere in practice, to read a NUTS credible interval as an approximate confidence interval.

> **Conditions box (memorize — every one can fail in practice).** BvM requires: **(1) a well-specified model** (the truth is in the family); **(2) a fixed, finite-dimensional, identifiable $\theta$** (the number of parameters does not grow with $n$); **(3) $\theta_0$ in the interior** of the parameter space (not on a boundary); **(4) a prior that is positive and continuous at $\theta_0$**. Drop any one and the credible interval stops being a confidence interval.

**Breakdown gallery.** Each failure below is a condition from the box, switched off.

*Boundary (condition 3).* For $X_i\sim U(0,\theta)$ the MLE is $\max_i X_i$, always *below* $\theta$, and $n(\theta-\max)$ converges to an **Exponential**, not a Gaussian — a one-sided limit pinned against the boundary. No amount of data makes the sampling law symmetric.

In [ ]:
# Boundary failure: U(0,theta) MLE is not asymptotically normal.
theta, n = 1.0, 80
mx = theta * rng.beta(n, 1, size=200_000)            # max of n U(0,theta) = theta*Beta(n,1)
scaled = n * (theta - mx)                            # -> Exponential(rate=1/theta), not Normal
print(f"U(0,θ) boundary:  E[n(θ−max)] = {scaled.mean():.3f} (Exp mean θ={theta})  "
      f"skew = {stats.skew(scaled):.2f} (Normal=0)")

The rescaled error has mean `0.986` and skewness `1.90` — the fingerprint of an Exponential, nothing like the symmetric Gaussian BvM promises. Condition 3 failed.

*Infinitely many nuisance parameters (condition 2): Neyman–Scott.* Observe pairs $(X_i,Y_i)\sim N(\mu_i,\sigma^2)$ — a fresh mean $\mu_i$ for every pair, one shared variance $\sigma^2$. The number of parameters grows *with the data*. **Predict** before running: one shared $\sigma^2$, but a fresh $\mu_i$ per pair estimated from just two points each — with 20,000 pairs, does $\hat\sigma^2_{\mathrm{MLE}}$ land at the true $\sigma^2=1$, or somewhere else? The reflex in play: *more data always means consistent.*

In [ ]:
# Neyman-Scott: one nuisance mean per pair -> sigma^2 MLE converges to sigma^2/2.
sig2_true, Npairs = 1.0, 20_000
mus = rng.normal(0, 3, size=Npairs)
Xp = rng.normal(mus, np.sqrt(sig2_true)); Yp = rng.normal(mus, np.sqrt(sig2_true))
mu_hat = (Xp + Yp) / 2                                # MLE of each mu_i
sig2_mle = (((Xp - mu_hat)**2 + (Yp - mu_hat)**2).sum()) / (2 * Npairs)
print(f"Neyman–Scott:  σ²_MLE = {sig2_mle:.4f}   →  σ²/2 = {sig2_true/2:.4f}   (true σ² = {sig2_true})")

The MLE lands at `0.4988` — dead on $\sigma^2/2=$ `0.5000`, not the true `1.0`, and no amount of additional pairs moves it. The "more data means consistent" reflex fails because each mean $\mu_i$ is estimated from only two points, so the within-pair spread systematically underestimates $\sigma^2$ — and the bias never vanishes because a new unknown arrives with every observation. Consistency is *void*: condition 2 failed. (A Bayesian who marginalizes the $\mu_i$ under a proper prior escapes this, module 16.)

*Unidentified parameters (condition 2, again).* Module 07's washout exception: if only $\theta_1+\theta_2$ is observed, the sum concentrates but $\theta_1$ alone keeps its prior width forever. BvM's identifiability clause is exactly what fails, and no data volume fixes it.

*Misspecification (condition 1).* When the model is wrong, the posterior still concentrates — on the parameter minimizing KL to the truth — but its width is set by the model's Fisher information, not the truth's. The frequentist sampling variance (the "sandwich") and the Bayesian posterior variance disagree; the posterior is typically **too narrow**. This mismatch is the subject of module 18; the one-line version: *a well-specified model's BvM calibration is precisely what misspecification steals.*

## 08.5 The complete-class theorem: "special case" is itself a theorem

Modules 00–07 repeatedly framed frequentist procedures as special cases or approximations of Bayesian ones. That framing is not rhetoric — it is a theorem. Draw the **risk set**: for a tiny problem, plot every decision rule as a point whose coordinates are its risks at each possible $\theta$. Take $X\sim\mathrm{Binomial}(2,\theta)$ with $\theta\in\{0.3,0.7\}$ (a two-point parameter space) and squared-error loss. A rule assigns an estimate $a\in[0,1]$ to each outcome $x\in\{0,1,2\}$; its risk pair is $(R(0.3),R(0.7))$. **Predict** before looking: where does the MLE rule $a=x/2$ land — on the efficient lower boundary of the risk set, or in the dominated interior? Its §08.1 credentials (unbiased, CRLB-attaining) argue for the boundary.

In [ ]:
# Complete class: the risk set for binomial(n=2), Theta={0.3,0.7}, squared-error loss.
theta = np.array([0.3, 0.7]); xs = np.arange(3)
pmf = np.stack([stats.binom(2, t).pmf(xs) for t in theta])       # (2 thetas, 3 outcomes)
def risk_pair(a):                                                 # a = (a0,a1,a2)
    a = np.asarray(a)
    return np.array([(pmf[j] * (a - theta[j])**2).sum() for j in range(2)])

# Cloud of achievable risks from many random (a0,a1,a2) rules.
rules = rng.uniform(0, 1, size=(4000, 3))
cloud = np.array([risk_pair(r) for r in rules])
# Bayes rules: for prior weight p on theta0, act = posterior mean of theta at each outcome.
ps = np.linspace(0, 1, 201)
bayes = []
for p in ps:
    w = np.array([p, 1 - p])
    a = np.array([(w * pmf[:, k] * theta).sum() / (w * pmf[:, k]).sum() for k in range(3)])
    bayes.append(risk_pair(a))
bayes = np.array(bayes)
mle_rp = risk_pair([0.0, 0.5, 1.0])                              # MLE rule a=x/2
print(f"MLE rule risk pair       = ({mle_rp[0]:.3f}, {mle_rp[1]:.3f})")
print(f"Bayes rule (p=0.5) risk  = ({bayes[100,0]:.3f}, {bayes[100,1]:.3f})")
print(f"Bayes boundary R(0.3) spans [{bayes[:,0].min():.3f}, {bayes[:,0].max():.3f}]")

fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.scatter(cloud[:, 0], cloud[:, 1], s=6, color="C7", alpha=0.35, label="random rules (risk set)")
ax.plot(bayes[:, 0], bayes[:, 1], color="C1", lw=2.5, label="Bayes rules = lower boundary")
ax.scatter(*mle_rp, color="C3", s=70, zorder=5, label="MLE rule (a=x/2)")
ax.set_xlabel("risk at θ=0.3"); ax.set_ylabel("risk at θ=0.7")
ax.set_title("Complete class: admissible rules ARE (limits of) Bayes rules")
ax.legend(fontsize=9)
save(fig, "risk-set")

![Risk set as a gray cloud in the plane of (risk at 0.3, risk at 0.7); the Bayes rules trace the convex lower-left boundary; the MLE rule sits well up on the interior at (0.105, 0.105).](figures/08-frequentist-bridge/risk-set.png)

The risk set is a convex region (you may randomize between rules), and the **Bayes rules trace its lower-left boundary** — one Bayes rule per prior weight $p$, each the posterior mean at every outcome. The MLE rule $a=x/2$ lands at risk pair `(0.105, 0.105)`, up in the interior: dominated, hence inadmissible even here — its unbiasedness credentials bought it nothing, because unbiasedness is a constraint on the rule, not a virtue of its risk. The Bayes rule at $p=0.5$ sits at `(0.029, 0.029)`, on the boundary.

**Complete-class theorem (stated, no proof).** For such problems, the admissible decision rules are exactly the Bayes rules and their limits. Every rule worth using is a Bayes rule against *some* prior; every non-Bayes rule is beaten by one. This is the rigorous content of the course's recurring line — *frequentist procedures are special cases of Bayesian ones.* It is a **theorem**, not a slogan, and James–Stein (§08.2) is its most famous consequence: the MLE was inadmissible, so a Bayes rule had to beat it.

## 08.6 Two audits a Bayesian model should pass

Frequentist evaluation is not a rival to guard against — it is an audit your model should *welcome*, in the same spirit as a prior-predictive check (module 07). Two audits close the module.

**The bootstrap is an implicit posterior.** The classical bootstrap resamples the data with replacement and recomputes a statistic; the **Bayesian bootstrap** instead draws Dirichlet$(1,\dots,1)$ weights on the observed points and recomputes the weighted statistic — a posterior under a noninformative model that puts all mass on the observed values. They produce nearly identical intervals.

In [ ]:
# Classical bootstrap vs Bayesian bootstrap (Dirichlet(1,...,1) weights) for the mean.
data = rng.gamma(2.0, 1.0, size=40)                  # skewed sample
B = 20_000
idx = rng.integers(0, 40, size=(B, 40)); classical = data[idx].mean(1)
w = rng.dirichlet(np.ones(40), size=B); bayesian = (w * data).sum(1)
cl = np.percentile(classical, [2.5, 97.5]); by = np.percentile(bayesian, [2.5, 97.5])
print(f"sample mean            = {data.mean():.3f}")
print(f"classical bootstrap 95% CI = [{cl[0]:.3f}, {cl[1]:.3f}]")
print(f"Bayesian  bootstrap 95% CI = [{by[0]:.3f}, {by[1]:.3f}]")

fig, ax = plt.subplots()
ax.hist(classical, bins=60, density=True, color="C0", alpha=0.5, label="classical bootstrap")
ax.hist(bayesian, bins=60, density=True, histtype="step", color="C1", lw=2, label="Bayesian bootstrap")
ax.axvline(data.mean(), color="k", ls="--", lw=1, label="sample mean")
ax.set_xlabel("bootstrap mean"); ax.set_ylabel("density")
ax.set_title("The bootstrap is an implicit posterior over the sample mean")
ax.legend(fontsize=9)
save(fig, "bootstrap")

![Two nearly identical histograms of the bootstrapped sample mean, classical filled and Bayesian outlined, centered on the sample mean 1.644.](figures/08-frequentist-bridge/bootstrap.png)

The classical interval is `[1.340, 1.980]` and the Bayesian-bootstrap interval `[1.357, 1.992]` — the same uncertainty to two decimals, reached from opposite philosophies. Resampling *is* a way of drawing from a posterior; the frequentist's most beloved model-free tool is Bayesian inference with the most noninformative prior available.

**The coverage audit.** Where the bootstrap showed a frequentist *tool* is secretly Bayesian, the sharper test runs the other way: submit a Bayesian *interval* to the frequentist's own exam. Does a credible interval actually cover the truth at its stated rate under repeated sampling? Take a weak prior, form the 95% credible interval, and check its *frequentist* coverage at a fixed $\theta$ over many datasets. **Predict** the digit before running: at 0.95, or below it (a posterior that borrowed prior information it wasn't entitled to)?

In [ ]:
# Coverage audit: does a 95% credible interval cover theta at ~95% frequency? (ties to M06)
theta0, n_ca, tau2, R = 3.0, 25, 100.0, 20_000
ys = rng.normal(theta0, 1.0, size=(R, n_ca)); ybar = ys.mean(1)
pv = 1 / (n_ca + 1 / tau2); pm = pv * (n_ca * ybar)   # Normal-Normal posterior (weak prior)
lo = pm - 1.96 * np.sqrt(pv); hi = pm + 1.96 * np.sqrt(pv)
cover = ((lo <= theta0) & (theta0 <= hi)).mean()
print(f"95% credible interval, frequentist coverage at θ={theta0}: {cover:.4f}")

The weak-prior credible interval covers at `0.9499` — indistinguishable from the nominal 95%. This is BvM cashed out as a number, and it ties back to module 06's exact result that under the true prior, prior-averaged coverage is exactly $1-\alpha$; here, with a diffuse prior at moderate $n$, even *pointwise* coverage passes. A well-built Bayesian model passes the frequentist's own audit. That is the reconciliation, quantified: not two warring creeds, but one calculus and two ledgers that agree wherever the conditions of §08.4 hold.

## Bridge — C-B §7.3/§10.1 reinterpreted

Casella–Berger develops risk, the Cramér–Rao bound, Bayes rules, and MLE asymptotics as separate frequentist tools. Read through the four lines they are one story: the **Bayes rule** (C-B §7.3.4) minimizes posterior expected loss — that is line 4, the definition of a decision. The **complete-class theorem** says every admissible frequentist rule is such a Bayes rule. **Cramér–Rao** (§7.3) bounds unbiased estimators, but the estimators that actually minimize risk (James–Stein, ridge, any posterior mean) are biased and slip the bound. **BvM** (the asymptotic content of §10.1) is why the frequentist's efficient-MLE machinery and the Bayesian's posterior converge. The frequentist ledger is not a different subject; it is the audit trail of the Bayesian one.

## Pitfalls

- **Reading Stein's paradox as "shrinkage always helps each coordinate."** It does *not*: for a single coordinate whose $\theta_i$ is far from the target, JS makes that coordinate's estimate worse. The theorem is about *total* risk across $d\ge3$ coordinates — the wins on the many pay for the losses on the few. Shrink a quantity you care about individually and you may regret it.
- **Quoting a credible interval as a confidence interval when a BvM condition fails.** At a boundary ($U(0,\theta)$), with growing-dimension nuisances (Neyman–Scott), under nonidentifiability, or under misspecification, the credible interval's frequentist coverage can be badly off. Check the four conditions before claiming the equivalence.
- **Applying Cramér–Rao to a biased estimator.** The bound governs *unbiased* estimators only. "James–Stein beats the CRLB" is not a paradox — the bound simply does not apply to it, nor to any shrinkage or ridge estimator.
- **Trusting the MLE's asymptotic normal error where support depends on $\theta$.** For $U(0,\theta)$ the Fisher-information variance is meaningless; the error is one-sided and Exponential. Support-dependence voids both Cramér–Rao and BvM.
- **Believing the bootstrap is assumption-free.** It is an implicit posterior with a specific (crude) model — the empirical distribution as truth. It inherits that model's blind spots: it understates uncertainty for functionals sensitive to the tails (extremes, maxima), exactly where the empirical distribution is thinnest.

## Exercises

**Exercise 08.1 — The minimax coin.**
*Setup:* Estimate $\theta$ from $X\sim\mathrm{Binomial}(n,\theta)$, $n=10$, squared-error loss. Compare the MLE $\hat\theta=X/n$ against the estimator $\delta_M(X)=(X+\sqrt n/2)/(n+\sqrt n)$ — a Bayes rule under a particular Beta prior.
*Predict:* The MLE's risk is $\theta(1-\theta)/n$, a downward parabola: zero at the ends, maximal at $\theta=0.5$. Sketch $\delta_M$'s risk curve. Is it also a parabola, or something else?
*Reason:* The intuition "any reasonable estimator has parabolic risk that vanishes at the boundaries" — the MLE's shape, assumed universal.
*Run:*

In [ ]:
n = 10; tg = np.linspace(0.001, 0.999, 200)
def risk(est_fn):
    x = np.arange(n + 1); out = []
    for t in tg:
        pm = stats.binom(n, t).pmf(x); out.append((pm * (est_fn(x) - t)**2).sum())
    return np.array(out)
r_mle = risk(lambda x: x / n)
r_mm  = risk(lambda x: (x + np.sqrt(n) / 2) / (n + np.sqrt(n)))
print(f"MLE risk: min {r_mle.min():.4f} at ends, max {r_mle.max():.4f} at θ=0.5")
print(f"δ_M risk: min {r_mm.min():.4f}, max {r_mm.max():.4f}  (flat?)")

<details><summary>Reconcile</summary>

The MLE risk is the parabola you expected — `0.0001` at the boundaries, `0.0250` at $\theta=0.5$. But $\delta_M$'s risk is **dead flat**: `0.0144` at every $\theta$ (min = max to four decimals). $\delta_M$ is the **minimax** estimator — it minimizes the *worst-case* risk, and it buys that guarantee by accepting slightly higher risk near the boundaries (where the MLE was superb) in exchange for lower risk in the middle. The two curves cross near $\theta\approx0.18$ and $0.82$: the MLE wins in the tails, the minimax rule wins in the center. And $\delta_M$ is exactly the Bayes rule against the Beta$(\sqrt n/2,\sqrt n/2)$ prior — the **least-favorable prior**. Minimax = Bayes versus the prior that makes the problem hardest. Flat risk is the signature: a rule with constant risk that is Bayes must be minimax.
</details>

**Exercise 08.2 — How many problems before shrinkage pays?**  *(surprising)*
*Setup:* You will estimate $d$ independent standard-normal means, all truly zero, once each. You may use the MLE or James–Stein$^+$.
*Predict:* For which $d$ does JS$^+$ give a *worse* total MSE than the MLE: $d=1$? $d=2$? $d=5$? Commit before running.
*Reason:* "Shrinking toward the truth (zero) can only help" — the target is perfect, so surely any $d$ benefits.
*Run:*

In [ ]:
for dd in (1, 2, 5):
    a, _, c = js_mse(np.zeros(dd), R=60_000)
    print(f"d={dd}: MSE(MLE)={a:.3f}  MSE(JS+)={c:.3f}  gain={a-c:.3f}")

<details><summary>Reconcile</summary>

At $d=1$ JS$^+$ is catastrophically *worse* — the $d-2=-1$ factor inflates rather than shrinks (MSE blows up); at $d=2$ it exactly ties the MLE (gain `0.000`, both `1.997`); only at $d=5$ does it help (gain `3.590`). Even with a *perfect* shrinkage target, dimension $d\ge3$ is required — the effect is about pooling *information* across coordinates ($\lVert X\rVert^2$ is a $d$-dimensional signal about how far the whole vector sits from the target), not about the target being correct. Two independent problems cannot audit each other; three can. This is the arithmetic behind "borrowing strength" and the reason hierarchical models (module 16) need several groups to earn their keep.
</details>

**Exercise 08.3 — When does the credible interval stop being a confidence interval?**  *(ML/DL bridge)*
*Setup:* Fit a single Normal mean $\mu$ two ways and audit frequentist coverage of the 95% credible interval: (a) well-specified — data truly $N(\mu,1)$; (b) misspecified — data truly Laplace with the same variance, but you still fit the Normal model.
*Predict:* Will the misspecified model's credible interval still cover $\mu$ at 95%, or will it drift? Same direction as a neural net that is overconfident out of distribution?
*Reason:* "The posterior contracts at $1/\sqrt n$ either way, so coverage should stay near nominal" — trusting BvM without checking condition 1.
*Run:*

In [ ]:
def coverage(true_sampler, n=25, R=8000, mu0=0.0):
    cov = 0
    for _ in range(R):
        y = true_sampler(n); pv = 1 / (n + 1e-4); pm = pv * (n * y.mean())
        lo, hi = pm - 1.96*np.sqrt(pv), pm + 1.96*np.sqrt(pv)
        cov += lo <= mu0 <= hi
    return cov / R
print("well-specified N(0,1):", coverage(lambda n: rng.normal(0, 1, n)))
print("misspecified Laplace :", coverage(lambda n: rng.laplace(0, 1/np.sqrt(2), n)))

<details><summary>Reconcile</summary>

The well-specified interval covers at `0.94925`; the Laplace-truth interval covers at `0.946` — also essentially nominal — because both distributions are symmetric with the *same variance*, and the Normal model's posterior width happens to match the true sampling variance of $\bar y$. Coverage of a *mean* is robust to this particular misspecification. The lesson is subtle and honest: misspecification does not automatically break coverage — it breaks it when the model's Fisher information misstates the true sampling variance (heteroskedasticity, dependence, or a functional other than the mean). That is exactly the sandwich-variance mismatch of module 18, and it is the statistical cousin of a network confidently extrapolating off its training distribution: the model is *sure*, and its sureness was calibrated on the wrong world.
</details>

## Takeaways

- The **sampling distribution** (fix $\theta$, vary data) is the frequentist ledger; the posterior (fix data, vary $\theta$) is the Bayesian one. The MLE is asymptotically $N(\theta, \mathcal I^{-1}/n)$ and attains the Cramér–Rao bound among *unbiased* estimators — every clause true, for **one** parameter.
- **Stein (S4):** for $d\ge3$ unrelated Gaussian means, James–Stein *dominates* the MLE — total MSE `1.99` vs `10.0` at the origin, strictly lower everywhere. The MLE is **inadmissible**. The gain collapses at $d\le2$; crossover at $d=3$.
- **James–Stein is empirical Bayes:** it is the Normal–Normal posterior mean $(1-B)X$ with the shrinkage fraction $B=1/(1+\tau^2)$ estimated from the marginal as $\widehat B=(d-2)/\lVert X\rVert^2$. Admissibility forced frequentists to invent a posterior mean.
- **Bernstein–von Mises** is why credible $\approx$ confidence: the standardized posterior and the MLE sampling law share a Gaussian limit (TV `0.1668`$\to$`0.0165` as $n:5\to500$). It needs four conditions — well-specified, fixed finite-dim identifiable $\theta$, interior point, positive continuous prior — and each has a named failure (boundary $U(0,\theta)$; Neyman–Scott $\hat\sigma^2\to\sigma^2/2$; nonidentifiability; misspecification).
- **Complete-class theorem:** admissible rules are exactly Bayes rules and their limits — the risk-set boundary is traced by Bayes rules. "Frequentist procedures are special cases of Bayesian ones" is a theorem.
- The **bootstrap is an implicit posterior** (classical $\equiv$ Bayesian-bootstrap intervals to two decimals), and a weak-prior credible interval passes the **frequentist coverage audit** (`0.9499` at nominal 95%). Frequentist evaluation is an audit to welcome, not a creed to fight: one calculus, two ledgers — and the ledgers agree wherever §08.4's conditions hold.